# 08. 입찰메이트 RFP RAG — Phase 5 근거 기반 답변·전체 요약 고도화

유영관 · 2026-11-11

이미 성공한 Phase 4 Q&A·비교와 인용 복원·분리 수정을 포함합니다. 이번에는 문서 전체에서 **11개 항목을 분리해 요약**하고, 금액·VAT·기간·예외 조건을 검토할 수 있게 합니다.

| 이번에 추가한 내용 | 확인 방법 |
| --- | --- |
| 사업명부터 평가기준까지 11개 항목 | 항목별 내용·인용·처리 상태 |
| 공고문 참조·미확인·충돌 후보 구분 | 서로 다른 후보와 근거를 보존 |
| 금액·VAT·기간·조건 진단 | 원문과 답변의 문자 표현 차이를 검토 대상으로 표시 |
| 실행 결과와 사람 검토 양식 저장 | 결과에 연결된 검토 양식으로 확인 |

**기본 Run All의 API 호출은 0회입니다.** 실제 답변은 아래 옵션을 켠 로컬 실행에서만 만듭니다.
전체 문서의 모든 청크를 처리해도 의미적 요약이 완전하다는 뜻은 아닙니다. 자동 인용 검사는 의미적 정답 판정과 구분합니다.
후속 대화 고도화, 정식 성능 평가, 운영 UI, RAG 외 업무 도구는 이후 범위입니다.


## 1. 실행 환경과 옵션

현재 잘 작동하는 **Phase 3의 `.venv/bin/python` 커널을 계속 사용해도 됩니다.**
VS Code 노트북 오른쪽 위 → Select Another Kernel → Python Environments에서 그 Python을 선택하세요.
Phase 5 폴더와 커널의 가상환경 폴더가 같을 필요는 없습니다.

아래 셀의 기본값 그대로 먼저 Run All 하세요. 실제 API를 사용할 때는 `ENABLE_OPENAI=True`와 실행할 기능의 옵션 하나를 `True`로 바꾸고 커널을 재시작해 다시 실행합니다.
API 키는 뒤의 로컬 비밀번호 입력창에서 입력하며, 노트북에 직접 적지 않습니다.


In [1]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import json
import os
import sys

PROJECT_ROOT = None  # 필요하면 압축을 푼 Phase 5 폴더의 Path를 지정
ENABLE_OPENAI = True
RUN_LIVE_QA = False
RUN_LIVE_SUMMARY = True

if PROJECT_ROOT is None:
    candidates = [Path.cwd(), Path.cwd().parent,
                  Path.cwd() / 'rfp_phase5_local_20261111',
                  Path.cwd() / 'output/rfp_phase5_local_20261111']
    ROOT = next((p.resolve() for p in candidates
                 if (p / 'rfp_phase5/runtime.py').is_file()
                 and (p / 'index/index_meta.json').is_file()), None)
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None or not (ROOT / 'rfp_phase5/runtime.py').is_file():
    raise FileNotFoundError('PROJECT_ROOT에 압축을 푼 Phase 5 폴더를 지정하세요.')
print('실행 중인 Python:', sys.executable)
print('Python 버전:', sys.version.split()[0])
print('프로젝트 경로:', ROOT)
stored = json.loads((ROOT / 'index/index_meta.json').read_text(encoding='utf-8'))['dependencies']
mismatches = []
for name, expected in stored.items():
    try:
        current = version(name)
    except PackageNotFoundError:
        current = '설치되지 않음'
    print(f'{name}: 저장된 버전={expected}, 현재 버전={current}, 일치={current == expected}')
    if current != expected:
        mismatches.append(name)
if mismatches:
    raise RuntimeError('저장 인덱스와 커널의 의존성 버전이 다릅니다. 기존에 성공한 Phase 3 .venv/bin/python 커널을 선택하세요. 새 환경이라면 README_PHASE5.md의 설치 명령을 실행하고 커널을 Restart하세요. 인덱스를 자동 재구축하지 않습니다.')
LIVE_REQUESTED = bool(RUN_LIVE_QA or RUN_LIVE_SUMMARY)
if LIVE_REQUESTED and not ENABLE_OPENAI:
    raise ValueError('실제 실행 옵션을 켜려면 ENABLE_OPENAI도 True여야 합니다.')
if LIVE_REQUESTED:
    try:
        sdk_version = version('openai')
    except PackageNotFoundError:
        raise RuntimeError('이 노트북 커널에 OpenAI SDK가 없습니다. 커널 경로의 Python으로 requirements-phase5.txt를 설치한 뒤 커널을 Restart하세요.') from None
    print('OpenAI SDK:', sdk_version)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('환경 버전 검사: 통과')
print('실효 API 활성화:', bool(ENABLE_OPENAI and LIVE_REQUESTED))
print('Q&A 실행 옵션:', RUN_LIVE_QA, '/ 전체 요약 실행 옵션:', RUN_LIVE_SUMMARY)
print('실행 옵션이 모두 False이면 API 호출 없이 원문·요약 계획만 점검합니다.')


실행 중인 Python: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/.venv/bin/python
Python 버전: 3.11.7
프로젝트 경로: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111
numpy: 저장된 버전=2.3.5, 현재 버전=2.3.5, 일치=True
scipy: 저장된 버전=1.17.0, 현재 버전=1.17.0, 일치=True
scikit-learn: 저장된 버전=1.8.0, 현재 버전=1.8.0, 일치=True
OpenAI SDK: 3.49.0
환경 버전 검사: 통과
실효 API 활성화: True
Q&A 실행 옵션: False / 전체 요약 실행 옵션: True
실행 옵션이 모두 False이면 API 호출 없이 원문·요약 계획만 점검합니다.


## 2. 원문·청크·인덱스 불러오기

기존 Phase 3의 원문과 검색 인덱스를 검사해서 불러옵니다. 기존 데이터와 평가 보고서는 수정하지 않습니다.
아래 실습은 이미 사용한 **dev 사례의 질문과 선택 문서만** 사용합니다. 정답은 생성 프롬프트에 넣지 않으며 holdout은 읽지 않습니다.


In [2]:
from rfp_phase5.runtime import bootstrap_phase5, make_components, save_run
from rfp_phase5.fields import FIELDS
from rfp_phase5.quality import audit_result
from rfp_phase5.review import build_review, assess_review

if 'app' in globals() and isinstance(app, dict) and app.get('index') is not None:
    app['index'].close()
app = bootstrap_phase5(ROOT)
index = app['index']
catalog = index.get_catalog()
catalog_by_id = {row['doc_id']: row for row in catalog}

def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

try:
    from IPython.display import Markdown, display
except ImportError:
    def display(value):
        print(value)
    def Markdown(value):
        return value

# 기존 dev 파일에서 질문·문서 선택만 복사합니다. 정답은 사용하지 않습니다.
example_ids = {'F001', 'C001', 'D001', 'U003', 'S002'}
examples = {}
with (ROOT / 'eval/questions.dev.jsonl').open(encoding='utf-8') as handle:
    for line in handle:
        item = json.loads(line)
        if item.get('case_id') in example_ids:
            examples[item['case_id']] = {
                'question': item['question'],
                'selected_doc_ids': list(item.get('context', {}).get('selected_doc_ids', []))}
        del item
if set(examples) != example_ids:
    raise ValueError('기존 dev 실습 사례가 모두 포함되어 있는지 확인하세요.')
print('고유 문서:', len(catalog), '/ 원본 파일 ID:', len(app['documents']), '/ 청크:', len(app['chunks']))
print('원문·청크·인덱스 검사: 통과')
print('Phase 5 요약 항목:', len(FIELDS))
print(', '.join(field['label'] for field in FIELDS))
print('holdout 질문·정답: 읽지 않음')


고유 문서: 99 / 원본 파일 ID: 100 / 청크: 18266
원문·청크·인덱스 검사: 통과
Phase 5 요약 항목: 11
사업명, 발주기관, 목적, 주요 범위·필수요건, 예산·VAT, 수행기간, 참가·수행자격, 입찰 제출서류, 제출기한, 제출방식, 평가기준
holdout 질문·정답: 읽지 않음


## 3. 문서 선택과 전체 요약 계획 — API 호출 없음

기본 Q&A는 이미 확인한 서울시 패션브랜드 예산 질문입니다.
기본 전체 요약은 18개 청크인 **인천공항시설관리(주) 연말정산 위탁운영 용역(S002)** 하나입니다.
다른 문서를 요약하려면 `SUMMARY_DOC_ID`를 아래 승인 목록의 ID로 바꾸세요.

`APPROVED_PUBLIC_DOC_IDS`에 지정된 공개 문서만 API로 전송할 수 있습니다. 비밀 문서는 현재 공개 문서 실습에 섞지 않습니다.
여기서 출력하는 byte는 입력 크기 제한이며 토큰 수나 달러 금액이 아닙니다. 모든 배치의 계획을 만든 뒤 호출 수·입력 상한을 다시 검사합니다.


In [3]:
APPROVED_PUBLIC_DOC_IDS = sorted({did for example in examples.values()
                                  for did in example['selected_doc_ids']})
QA_SELECTED_DOC_IDS = examples['F001']['selected_doc_ids']
QA_QUESTION = examples['F001']['question']
QA_FEATURE = 'qa'  # 'comparison'이면 두 개 이상 문서를 QA_SELECTED_DOC_IDS에 선택
SUMMARY_DOC_ID = examples['S002']['selected_doc_ids'][0]

for did in APPROVED_PUBLIC_DOC_IDS:
    row = catalog_by_id[did]
    print(did, row['file_name'])
if not set(QA_SELECTED_DOC_IDS) <= set(APPROVED_PUBLIC_DOC_IDS):
    raise ValueError('Q&A 선택 문서는 공개 문서 승인 목록에 있어야 합니다.')
if SUMMARY_DOC_ID not in APPROVED_PUBLIC_DOC_IDS:
    raise ValueError('요약 선택 문서는 공개 문서 승인 목록에 있어야 합니다.')
planner = make_components(app, doc_ids=APPROVED_PUBLIC_DOC_IDS, enabled=False)
summary_plan = planner['summarizer'].plan(SUMMARY_DOC_ID)
assert [chunk['chunk_id'] for batch in summary_plan['batches'] for chunk in batch['chunks']] == summary_plan['source_chunk_ids']
print('요약할 문서:', catalog_by_id[SUMMARY_DOC_ID]['file_name'])
show_json({key: summary_plan.get(key) for key in [
    'chunk_count', 'map_calls', 'total_request_bytes', 'max_map_calls',
    'max_batch_bytes', 'can_execute', 'block_reasons',
    'llm_max_calls', 'llm_max_total_input_bytes',
    'fresh_session_uncached_within_caps', 'fresh_session_uncached_block_reasons']})
print('실제 실행 시 요청 전체의 남은 세션 상한과 캐시를 재검사합니다.')
print('요약 계획의 원문 청크 순서·개수 검사: 통과')


161LuHCu_ZzUIDue5yGL_rRKMUe10FZ38 한국관광공사_2026_AI 기반 관광 B2C 플랫폼 통합 ISP 수립_제안요청서.hwp
1Eq8gIdUnnLRbbEBiaU29wYji2YglNNuu 인천공항시설관리(주)_2026_2026년도 귀속 연말정산 위탁운영 용역_과업내용서.hwp
1G2at8Un7y8JIn5awuVxqV7OOKIj7zT4B 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp
1MxTCLhVvegbCCvqonlf95yO6pr4fYgaJ 한국주택금융공사_2021_부동산 등기정보 시스템 구축_제안요청서.hwp
1Uy4SDf7e_NFKK9CbjL9uVO4ta_MQdeTt 서울특별시_2026_2027년 글로벌 패션브랜드 육성 용역_제안요청서.hwpx
요약할 문서: 인천공항시설관리(주)_2026_2026년도 귀속 연말정산 위탁운영 용역_과업내용서.hwp
{
  "chunk_count": 18,
  "map_calls": 1,
  "total_request_bytes": 31301,
  "max_map_calls": 30,
  "max_batch_bytes": 35000,
  "can_execute": true,
  "block_reasons": [],
  "llm_max_calls": 30,
  "llm_max_total_input_bytes": 600000,
  "fresh_session_uncached_within_caps": true,
  "fresh_session_uncached_block_reasons": []
}
실제 실행 시 요청 전체의 남은 세션 상한과 캐시를 재검사합니다.
요약 계획의 원문 청크 순서·개수 검사: 통과


## 4. 실제 실행 준비

실제 실행 옵션이 켜진 경우에만 로컬 입력창에서 키를 받습니다. 키가 이미 커널 환경변수에 있으면 다시 묻지 않습니다.
설정 파일의 기본 `enabled=False`와 혼동하지 않도록 아래에는 **현재 실행의 실효 활성화 값**을 표시합니다.
기본 모델은 기존과 같은 `gpt-5.4-mini`이며, 자동 재시도나 대체 모델 호출은 하지 않습니다.

Q&A 한 번과 전체 요약은 호출 수가 다릅니다. 전체 요약은 위 계획의 `map_calls`만큼 요청할 수 있습니다.
상한에 걸리면 입력을 조용히 자르지 않고 실행을 차단합니다. 상한 조정은 `configs/phase5.json`에서 하고 커널을 재시작합니다.


In [4]:
if ENABLE_OPENAI and LIVE_REQUESTED and not os.environ.get('OPENAI_API_KEY', '').strip():
    from getpass import getpass
    local_key = getpass('OpenAI API 키 (화면에 표시되지 않음): ').strip()
    if not local_key:
        raise ValueError('API 키가 입력되지 않았습니다.')
    os.environ['OPENAI_API_KEY'] = local_key
    del local_key
components = make_components(app, doc_ids=APPROVED_PUBLIC_DOC_IDS,
                             enabled=bool(ENABLE_OPENAI and LIVE_REQUESTED))
print('생성 모델:', components['llm'].config.model)
print('실효 API 활성화:', components['llm'].config.enabled)
print('API 키 설정 여부:', bool(os.environ.get('OPENAI_API_KEY', '').strip()))
print('승인된 공개 문서 수:', len(components['approved_sources']))
show_json(components['llm'].usage_report())
qa_result = None
summary_result = None
saved_runs = []


생성 모델: gpt-5.4-mini
실효 API 활성화: True
API 키 설정 여부: True
승인된 공개 문서 수: 5
{
  "api_calls": 0,
  "live_api_calls": 0,
  "mock_calls": 0,
  "cache_hits": 0,
  "failed_attempts": 0,
  "attempted_input_bytes": 0,
  "reserved_max_output_tokens": 0,
  "reported_input_tokens": 0,
  "reported_output_tokens": 0,
  "reported_cached_input_tokens": 0,
  "reported_total_tokens": 0,
  "usage_unknown_attempts": 0,
  "request_latency_ms": 0.0,
  "max_calls": 30,
  "max_total_input_bytes": 600000,
  "execution_mode": "live_openai",
  "enabled": true,
  "budget_unit": "UTF-8 serialized request bytes; not token count or currency"
}


## 5. 근거 기반 Q&A와 보존 진단

위의 `QA_QUESTION`, `QA_SELECTED_DOC_IDS`를 바꾸면 직접 질문할 수 있습니다.
답변의 인용문이 원문에 있는지 검사하고, 숫자·단위·VAT·조건 표현 차이를 검토 대상으로 남깁니다.
진단 플래그가 없다는 이유만으로 답변이 정답이라고 판정하지 않습니다.
기존 Phase 4의 인용 복원·분리도 적용됩니다. 출처 검증에 실패한 답변은 표시하지 않습니다.


In [5]:
if RUN_LIVE_QA:
    components['qa'].select_documents(QA_SELECTED_DOC_IDS)
    qa_result = components['qa'].ask(QA_QUESTION, feature=QA_FEATURE)
    display(Markdown(qa_result.get('markdown', '답변 출력이 없습니다.')))
    show_json({key: qa_result.get(key) for key in ['status', 'execution_status', 'answer_generated', 'error_detail', 'errors']})
    qa_audit = audit_result(qa_result)
    show_json(qa_audit)
    qa_run = save_run(ROOT, qa_result, operation='qa', usage=components['llm'].usage_report(),
                      pricing=app['phase5_config']['pricing'])
    saved_runs.append(qa_run)
    print('이번 Q&A 결과·진단·검토 양식:', qa_run)
else:
    print('Q&A 실제 생성: 미실행. ENABLE_OPENAI와 RUN_LIVE_QA를 True로 바꾸면 실행합니다.')


Q&A 실제 생성: 미실행. ENABLE_OPENAI와 RUN_LIVE_QA를 True로 바꾸면 실행합니다.


## 6. 11개 항목의 전체 문서 요약

선택 문서의 모든 청크를 순서대로 배치에 넣고, 원문 출처가 검증된 주장만 항목별로 합칩니다.

| 항목 상태 | 의미 |
| --- | --- |
| 근거 확보 | 처리한 입력에서 인용 가능한 내용을 찾음 |
| 공고문 참조 | 원문이 명시적으로 공고문 확인을 요구함 |
| 충돌 후보 | 서로 다른 내용을 보존하고 검토가 필요함 |
| 처리한 입력에서 미확인 | 현재 처리 범위에서 근거를 확보하지 못함 |
| 미처리 | 실행되지 않았거나 해당 범위 처리가 끝나지 않음 |

서로 다른 값이 나와도 파일명 연도나 다수결로 정답을 고르지 않습니다.
부분 입력의 미확인 사항은 범위를 붙여 남깁니다. 다른 배치에서 같은 항목이 발견돼도 세부 조건이 모두 해결됐다고 자동 단정하지 않습니다.


In [6]:
if RUN_LIVE_SUMMARY:
    summary_result = components['summarizer'].summarize(SUMMARY_DOC_ID)
    display(Markdown(summary_result.get('markdown', '요약 출력이 없습니다.')))
    show_json({key: summary_result.get(key) for key in [
        'status', 'execution_status', 'answer_generated',
        'input_chunk_coverage', 'validated_chunk_coverage', 'semantic_quality_verified', 'errors']})
    show_json([{'field_id': row['field_id'], '항목': row['label'], '상태': row['state'],
                '근거 주장 수': len(row['claims'])} for row in summary_result.get('fields', [])])
    summary_audit = audit_result(summary_result)
    show_json(summary_audit)
    summary_run = save_run(ROOT, summary_result, operation='summary',
                           usage=components['llm'].usage_report(),
                           pricing=app['phase5_config']['pricing'])
    saved_runs.append(summary_run)
    print('이번 요약 결과·진단·검토 양식:', summary_run)
else:
    print('전체 문서 요약 실제 생성: 미실행. ENABLE_OPENAI와 RUN_LIVE_SUMMARY를 True로 바꾸면 실행합니다.')


| 항목 | 내용 | 확인 상태 |
| --- | --- | --- |
| 사업명 | 2026년도 귀속 연말정산 위탁운영 용역 [1] | 근거 추출됨 · 검토 필요 |
| 발주기관 | 이번에 처리한 입력에서 근거를 확보하지 못했습니다. | 처리한 입력에서 미확인 |
| 목적 | 세무 전문업체를 통한 연말정산 교육 및 신고자료 검토 업무 대행을 통한 연말정산 업무의 효율적 추진, 개정 세법에 대한 정보자료 제공, 연말정산 방법 교육 및 상담을 통한 연말정산 신고오류 최소화 [2][3] | 근거 추출됨 · 검토 필요 |
| 주요 범위·필수요건 | 연말정산 안내문 및 더존 IU 입력 매뉴얼 작성 및 배포, IU 반영사항 오류 검증 및 프로그램 테스트, 원천세 신고 자료 및 지급명세서 비교 검증, 임직원 상담 등 문의 응대, 인천공항시설관리\(주\) 내 상주하여 연말정산 서류 접수, IU 입력보조, 서류·전산대조의 검토, 오류 검증 등, 휴가, 휴직, 정년퇴직자 등에 따른 임직원 부재시 부재자 서류취합 및 자료등록, 중도입사자\(전직장 소득합산\) 및 중도퇴직자 정산자료 검토지원, 전산매체 신고 시 오류 검증 등, 지급명세서 제출 및 오류 시 수정 제출, 주요 문의내용 및 답변서 제출, 2026년 귀속 연말정산 결과보고서 제출, 연말정산 증빙서류 보관용 서류박스\(지정 규격\) 구매, 편철 및 최종 제출, 국세청으로부터 부당 공제자 통보 시 후속처리 절차 지원\(재신고·재계산 처리\), 개인정보보호법 등 관련 법령 및 우리 회사의 사내 개인정보처리 규정 준수, 개인정보처리에 따른 보안서약서 제출 및 과업 완료 후 관련 수집 자료의 안전한 파기 및 반납 처리 [4][5][6][7][8][9][10][11][12][13][14][15][16][17] | 근거 추출됨 · 검토 필요 |
| 예산·VAT | 사업예산은 57,246,000원\(부가가치세 제외\) [18] | 근거 추출됨 · 검토 필요 |
| 수행기간 | 계약체결일 ～ 2027\. 3\. 17 [19] | 근거 추출됨 · 검토 필요 |
| 참가·수행자격 | 총괄책임자는 세무사 자격 소지자로 연말정산 업무 3년 이상의 유경험자로 하며, 인천공항시설관리\(주\) 상주 근무인력은 연말정산 업무 2년 이상의 유경험자 3명 이상으로 한다\. [20] | 근거 추출됨 · 검토 필요 |
| 입찰 제출서류 | 이번에 처리한 입력에서 근거를 확보하지 못했습니다. | 처리한 입력에서 미확인 |
| 제출기한 | 이번에 처리한 입력에서 근거를 확보하지 못했습니다. | 처리한 입력에서 미확인 |
| 제출방식 | 이번에 처리한 입력에서 근거를 확보하지 못했습니다. | 처리한 입력에서 미확인 |
| 평가기준 | 이번에 처리한 입력에서 근거를 확보하지 못했습니다. | 처리한 입력에서 미확인 |

원문 근거:

1. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 10 … BodyText/Section0 / paragraph 11 — “2026년도 귀속 연말정산 위탁운영 용역”
2. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 56 … BodyText/Section0 / paragraph 85 — “세무 전문업체를 통한 연말정산 교육 및 신고자료 검토 업무 대행을 통한 연말정산 업무의 효율적 추진”
3. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 56 … BodyText/Section0 / paragraph 85 — “개정 세법에 대한 정보자료 제공, 연말정산 방법 교육 및 상담을 통한 연말정산 신고오류 최소화”
4. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “연말정산 안내문 및 더존 IU 입력 매뉴얼 작성 및 배포”
5. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “IU 반영사항 오류 검증 및 프로그램 테스트”
6. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “원천세 신고 자료 및 지급명세서 비교 검증”
7. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “인천공항시설관리\(주\) 내 상주하여 연말정산 서류 접수, IU 입력보조, 서류·전산대조의 검토, 오류 검증 등”
8. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “휴가, 휴직, 정년퇴직자 등에 따른 임직원 부재시 부재자 서류취합 및 자료등록”
9. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “중도입사자\(전직장 소득합산\) 및 중도퇴직자 정산자료 검토지원”
10. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “전산매체 신고 시 오류 검증 등”
11. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “지급명세서 제출 및 오류 시 수정 제출”
12. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “주요 문의내용 및 답변서 제출”
13. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “2026년 귀속 연말정산 결과보고서 제출”
14. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “연말정산 증빙서류 보관용 서류박스\(지정 규격\) 구매, 편철 및 최종 제출”
15. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “국세청으로부터 부당 공제자 통보 시 후속처리 절차 지원\(재신고·재계산 처리\)”
16. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “개인정보보호법 등 관련 법령 및 우리 회사의 사내 개인정보처리 규정 준수”
17. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 88 … BodyText/Section0 / paragraph 112 — “개인정보처리에 따른 보안서약서 제출 및 과업 완료 후 관련 수집 자료의 안전한 파기 및 반납 처리”
18. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 56 … BodyText/Section0 / paragraph 85 — “사업예산 : 57,246,000원\(부가가치세 제외\)”
19. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 56 … BodyText/Section0 / paragraph 85 — “기간 : 계약체결일 ～ 2027\. 3\. 17”
20. 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 155 … BodyText/Section0 / paragraph 181 — “총괄책임자는 세무사 자격 소지자로 연말정산 업무 3년 이상의 유경험자로 하며, 인천공항시설관리\(주\) 상주 근무인력은 연말정산 업무 2년 이상의 유경험자 3명 이상으로 한다\.”

추가 확인 사항:

- 부분 입력 map\-0001에서 미확인: issuing\_agency
- 부분 입력 map\-0001에서 미확인: submission\_documents
- 부분 입력 map\-0001에서 미확인: submission\_deadline
- 부분 입력 map\-0001에서 미확인: submission\_method
- 부분 입력 map\-0001에서 미확인: evaluation

입력 처리율과 인용문 출처를 검사했습니다. 항목의 정확성·조건 보존·요약 완전성은 사람이 원문과 대조해야 합니다.

{
  "status": "partial",
  "execution_status": "completed",
  "answer_generated": true,
  "input_chunk_coverage": 1.0,
  "validated_chunk_coverage": 1.0,
  "semantic_quality_verified": false,
  "errors": []
}
[
  {
    "field_id": "project_name",
    "항목": "사업명",
    "상태": "evidence_found",
    "근거 주장 수": 1
  },
  {
    "field_id": "issuing_agency",
    "항목": "발주기관",
    "상태": "not_found_in_processed_input",
    "근거 주장 수": 0
  },
  {
    "field_id": "purpose",
    "항목": "목적",
    "상태": "evidence_found",
    "근거 주장 수": 1
  },
  {
    "field_id": "scope_requirements",
    "항목": "주요 범위·필수요건",
    "상태": "evidence_found",
    "근거 주장 수": 1
  },
  {
    "field_id": "budget_vat",
    "항목": "예산·VAT",
    "상태": "evidence_found",
    "근거 주장 수": 1
  },
  {
    "field_id": "duration",
    "항목": "수행기간",
    "상태": "evidence_found",
    "근거 주장 수": 1
  },
  {
    "field_id": "eligibility",
    "항목": "참가·수행자격",
    "상태": "evidence_found",
    "근거 주장 수": 1
  },
  {
    "field_id": "submission_documents",

## 7. 사람 검토와 이번 실행 상태

각 실행 폴더의 `human_review.json`은 결과·원문에 연결된 검토 양식입니다.
`reviewer`에 검토자 이름을 적고 원문을 직접 확인한 뒤 `rows` 안의 `supported`(주장을 뒷받침하는가), `complete`(필수 내용이 빠지지 않았는가),
`conditions_preserved`(예외·조건을 보존했는가)를 `true` 또는 `false`로 기입합니다. 미검토 값은 `null`로 둡니다.
`notes`에는 검토 내용을 적을 수 있습니다. fingerprint·문서 ID·항목 ID·주장과 인용은 수정하지 마세요. 다른 답변의 검토 양식을 재사용하면 연결 검사에서 거절됩니다.

검토 결과를 확인하려면 아래 셀의 `REVIEW_PATH`에 수정한 `human_review.json` 경로를 지정하고 이 셀만 실행하세요.
아래는 준비된 코드와 실제 모델 품질을 구분합니다. 모의 응답이나 실행하지 않은 항목을 완료로 집계하지 않습니다.


In [7]:
REVIEW_PATH = None  # 사람 검토 후 해당 human_review.json의 Path를 지정
latest_result = summary_result if summary_result is not None else qa_result
if REVIEW_PATH is not None:
    review_path = Path(REVIEW_PATH).expanduser().resolve()
    reviewed_result = json.loads((review_path.parent / 'result.json').read_text(encoding='utf-8'))
    human_review = json.loads(review_path.read_text(encoding='utf-8'))
    show_json(assess_review(reviewed_result, human_review))
elif latest_result is not None:
    pending_review = build_review(latest_result, case_id='notebook_current_run')
    show_json(assess_review(latest_result, pending_review))
else:
    print('사람 검토: 생성 결과가 없어 미실행')
usage = components['llm'].usage_report()
print('인덱스 환경 버전 검사: 통과')
print('전체 문서 요약 계획의 청크 순서·개수 검사: 통과')
print('Q&A 실제 생성 상태:', qa_result.get('execution_status') if qa_result is not None else '미실행')
print('전체 요약 실제 생성 상태:', summary_result.get('execution_status') if summary_result is not None else '미실행')
print('실제 API 호출 시도 수:', usage['live_api_calls'])
print('로컬 캐시 적중 수:', usage['cache_hits'])
print('모의 호출 수:', usage['mock_calls'])
print('의미적 정답·요약 완전성: 사람 검토와 별도 평가 필요')
for run_path in saved_runs:
    print('이번 실행 기록:', run_path)
print('Phase 5 실제 품질을 판단할 때는 요약의 11개 항목, 진단 플래그, 사람 검토를 함께 확인합니다.')


{
  "status": "review_pending",
  "live_readiness": false,
  "errors": [],
  "pending_judgements": [
    {
      "row_id": "project_name",
      "judgement": "supported"
    },
    {
      "row_id": "project_name",
      "judgement": "complete"
    },
    {
      "row_id": "project_name",
      "judgement": "conditions_preserved"
    },
    {
      "row_id": "issuing_agency",
      "judgement": "supported"
    },
    {
      "row_id": "issuing_agency",
      "judgement": "complete"
    },
    {
      "row_id": "issuing_agency",
      "judgement": "conditions_preserved"
    },
    {
      "row_id": "purpose",
      "judgement": "supported"
    },
    {
      "row_id": "purpose",
      "judgement": "complete"
    },
    {
      "row_id": "purpose",
      "judgement": "conditions_preserved"
    },
    {
      "row_id": "scope_requirements",
      "judgement": "supported"
    },
    {
      "row_id": "scope_requirements",
      "judgement": "complete"
    },
    {
      "row_id": "scope_re

In [8]:
show_json({
    key: summary_result.get(key)
    for key in [
        "status", "execution_status",
        "error_detail", "error_diagnostics", "usage"
    ]
})

{
  "status": "partial",
  "execution_status": "completed",
  "error_detail": null,
  "error_diagnostics": null,
  "usage": {
    "api_calls": 1,
    "live_api_calls": 1,
    "mock_calls": 0,
    "input_tokens": 8752,
    "output_tokens": 1923,
    "total_tokens": 10675,
    "usage_unknown_attempts": 0,
    "cached_input_tokens": 8576,
    "scope": "Counters for this summary invocation; failed attempts may have unknown token usage."
  }
}
